<a href="https://colab.research.google.com/github/kamrynphipps/Sephora-Data-Strategy-Analysis/blob/main/Sephora_Cross_Product_Compatibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import pandas as pd

In [ ]:
!pip install flashtext

  Preparing metadata (setup.py) ... done
  Created wheel for flashtext: filename=flashtext-2.7-py2.py3-none-any.whl size=9300 sha256=4567e3b3695c9dc6869e97ee898946ee7155a2b2fdadf14d5acc9689187985cb
  Stored in directory: /root/.cache/pip/wheels/8c/24/da/4d994d7a27cfc73a4e513a669fbeec4a71f871fe245a81977f
Successfully built flashtext


In [ ]:
files = ["/reviews_0-250.csv", "/reviews_250-500.csv", "/reviews_500-750.csv", "/reviews_750-1250.csv", "/reviews_1250-end.csv"]

In [ ]:
df_reviews = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in files],
    ignore_index=True
)

In [ ]:
df_products = pd.read_csv("/product_info.csv")

In [ ]:
df = df_reviews.merge(df_products, on="product_id", how="left")

In [ ]:
df = df.sample(n=50000, random_state=42)

In [ ]:
# Remove short / generic names
product_list = (df_products["product_name"].dropna().str.lower().str.strip())

# Keep only meaningful product names (at least 2 words, > 5 chars)
product_list = [
    p for p in product_list
    if len(p) > 5 and len(p.split()) >= 2]

brand_list = df_products['brand_name'].str.lower().unique().tolist()

In [ ]:
category_words = {
    "lipstick", "cleanser", "cream", "serum",
    "lip balm", "lip gloss", "eye cream",
    "foundation", "concealer", "bronzer", "blush"
}

product_list = [
    p for p in product_list
    if p not in category_words
]

In [ ]:
from flashtext import KeywordProcessor

keyword_processor = KeywordProcessor(case_sensitive=False)

product_list = (
    df_products["product_name"]
    .dropna()
    .str.lower()
    .str.strip()
    .unique()
)

# remove generic/noisy names before adding to matcher
product_list = [
    p for p in product_list
    if len(p) > 5 and len(p.split()) >= 2
]

keyword_processor.add_keywords_from_list(product_list)

In [ ]:
def extract_mentions(text):
    if pd.isna(text):
        return []
    return keyword_processor.extract_keywords(text)

df["mentioned_products"] = df["review_text"].apply(extract_mentions)

In [ ]:
from itertools import combinations

pairs = []

for products in df["mentioned_products"]:
    unique_products = list(set(products))  # (remove duplicates)
    if len(unique_products) > 1:
        pairs.extend(combinations(unique_products, 2))

co_df = pd.DataFrame(pairs, columns=["product_A", "product_B"])
co_counts = co_df.value_counts().reset_index(name="count")

In [ ]:
co_counts.head(50)

,product_A,product_B,count
0,face cream,eye cream,23
1,lip gloss,lip balm,16
2,lip sleeping mask,lip balm,15
3,eye cream,the one,11
4,face cream,glow drops,9
5,the lip balm,lip balm,9
6,lip liner,lip balm,9
7,the one,lip balm,7
8,rosebud salve,lip balm,6
9,face cream,hyaluronic serum,6


In [ ]:
df_products.columns

Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'variation_type', 'variation_value',
       'variation_desc', 'ingredients', 'price_usd', 'value_price_usd',
       'sale_price_usd', 'limited_edition', 'new', 'online_only',
       'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category',
       'secondary_category', 'tertiary_category', 'child_count',
       'child_max_price', 'child_min_price'],
      dtype='object')

In [ ]:
print('Columns of df_products:')
display(df_products.columns)
print('\nFirst 5 rows of df_products:')
display(df_products.head())

Columns of df_products:


Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'variation_type', 'variation_value',
       'variation_desc', 'ingredients', 'price_usd', 'value_price_usd',
       'sale_price_usd', 'limited_edition', 'new', 'online_only',
       'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category',
       'secondary_category', 'tertiary_category', 'child_count',
       'child_max_price', 'child_min_price'],
      dtype='object')


First 5 rows of df_products:


,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,NaN,NaN,NaN,...,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,85.0,30.0
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.2500,16.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
3,P473660,Kasbah Eau de Parfum,6342,19-69,3018,4.4762,21.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
4,P473658,Purple Haze Eau de Parfum,6342,19-69,2691,3.2308,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
